# 05 — SARIMA and SARIMAX Models
## Food Price Forecasting for Stable Commodities in Nigeria
### TS Academy Data Science Capstone 2026

---

## Purpose of This Notebook

This notebook builds and evaluates two statistical forecasting
models — SARIMA and SARIMAX — on the master dataset produced
by the Data Cleaning and Merging pipeline and analysed in the
EDA notebook.

SARIMA and SARIMAX are the classical statistical baseline models
for time series forecasting. They are built first because they
establish the performance benchmark that every other model —
XGBoost and Prophet — must beat to justify their added complexity.

## What This Notebook Covers

1. **Feature Engineering and Data Preparation** — log
   transformation, structural break indicators, temporal
   split and evaluation functions
2. **Naive Baseline Model** — the simplest possible forecast
   that all models must outperform
3. **SARIMA** — univariate statistical baseline using
   price history only
4. **SARIMAX** — SARIMA extended with exogenous economic
   features confirmed by EDA
5. **SARIMA vs SARIMAX Comparison** — determining whether
   external features genuinely improve predictions

## Key EDA Findings That Drive Every Decision Here

- All 7 commodities require d=1 differencing — confirmed
  by ADF test across 65 state-commodity combinations
- Seasonal period s=12 confirmed for all commodities
- June 2023 fuel subsidy removal is the dominant structural
  break in the dataset
- price_lag1 is the strongest predictor at r=0.968
- exchange_rate and pms_price are the strongest external
  predictors at r=0.63 and r=0.61 respectively
- Rice imported Jigawa excluded — only 1 month of data

## **Step 0 - Environment Setup**

### Purpose

Before any analysis begins we mount the Google Drive,
import all required libraries and load the master dataset
into this notebook. Every subsequent step depends on this
cell running successfully first.

In [ ]:
#Mounting my Drive
#connect my drive to colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import warnings
warnings.filterwarnings('ignore')

# statsmodels -- SARIMA and SARIMAX
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# scikit-learn -- evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# plot style
plt.style.use('seaborn-v0_8-whitegrid')

print('All libraries imported successfully')

All libraries imported successfully


In [ ]:
# load the master dataset
df = pd.read_csv('/content/drive/MyDrive/price forecasting project data(cleaned)/master_dataset_clean.csv')

# convert date column
df['date'] = pd.to_datetime(df['date']).dt.to_period('M')

# verify it loaded correctly
print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'States: {df["state"].nunique()}')
print(f'Commodities: {df["commodity"].nunique()}')
print(f'\nCommodities: {df["commodity"].unique()}')

Shape: (2846, 18)
Date range: 2017-01 to 2024-12
States: 13
Commodities: 7

Commodities: ['Maize (white)' 'Maize (yellow)' 'Rice (imported)' 'Rice (local)' 'Yam'
 'Beans (white)' 'Tomatoes']


## **Step 1 — Feature Engineering and Data Preparation**

### Purpose
Before any model is trained the dataset must be transformed
into the exact format SARIMA and SARIMAX expect. This step
creates all additional features identified in the EDA and
prepares the temporal train, validation and test splits.

Every transformation here is justified by a specific EDA
finding — nothing is done arbitrarily.

### What This Step Covers
1. **Log transform price_ngn** — target variable preparation
2. **Structural break indicators** — subsidy removal and COVID
3. **Cyclical month encoding** — sine and cosine for seasonality
4. **Lagged rainfall features** — 3 to 6 month agricultural lag
5. **Log transform import volume** — handling extreme right skew
6. **Harvest season interaction term** — isolating genuine effect
7. **Temporal train/validation/test split** — walk-forward only
8. **Evaluation functions** — RMSE, MAPE and MAE in Naira scale

In [ ]:
# ── Step 1: Feature Engineering and Data Preparation ──

# create a working copy -- never modify the original dataframe
df_model = df.copy()

# convert date to timestamp for plotting and time comparisons
# Period format cannot be used in arithmetic comparisons
df_model['date_ts'] = df_model['date'].dt.to_timestamp()

# 1a -- log transform the target variable
# price_ngn is severely right-skewed (skewness > 2.0 confirmed in EDA)
# log1p is used instead of log to safely handle any zero values
df_model['log_price'] = np.log1p(df_model['price_ngn'])

print('Log transform applied successfully')
print(f'Original price range: ₦{df_model["price_ngn"].min():.0f} to ₦{df_model["price_ngn"].max():.0f}')
print(f'Log price range: {df_model["log_price"].min():.2f} to {df_model["log_price"].max():.2f}')

Log transform applied successfully
Original price range: ₦30 to ₦6358
Log price range: 3.43 to 8.76


In [ ]:
# 1b -- structural break and event indicators
# June 2023 subsidy removal is the single most impactful event
# in the dataset -- confirmed by every chart in EDA Step 5

df_model['subsidy_removed'] = (
    df_model['date_ts'] >= '2023-06-01'
).astype(int)

# COVID period indicator -- March 2020 to December 2021
df_model['covid_period'] = (
    (df_model['date_ts'] >= '2020-03-01') &
    (df_model['date_ts'] <= '2021-12-31')
).astype(int)

# year and month features
df_model['year']  = df_model['date_ts'].dt.year
df_model['month'] = df_model['date_ts'].dt.month

# cyclical month encoding
# raw month numbers treat December (12) and January (1) as
# 11 units apart -- but they are only 1 month apart in reality
# sine and cosine encoding places them on a circle so the
# model correctly understands their proximity
df_model['month_sin'] = np.sin(2 * np.pi * df_model['month'] / 12)
df_model['month_cos'] = np.cos(2 * np.pi * df_model['month'] / 12)

print('Structural break indicators created successfully')
print(f'Subsidy removed observations: {df_model["subsidy_removed"].sum()}')
print(f'COVID period observations: {df_model["covid_period"].sum()}')

Structural break indicators created successfully
Subsidy removed observations: 225
COVID period observations: 1047


In [ ]:
# 1c -- lagged rainfall features
# same-month rainfall correlation with price = 0.05 (near zero)
# confirmed in EDA section 4.1 -- agricultural effect operates
# through a 3 to 6 month lag not same month

# CRITICAL: always sort by state and commodity before creating lags
# without sorting, lag values would bleed across different
# state-commodity combinations producing silent errors
df_model = df_model.sort_values(
    ['state', 'commodity', 'date_ts']
).reset_index(drop=True)

df_model['rainfall_lag3'] = df_model.groupby(
    ['state', 'commodity'])['rainfall_mm'].shift(3)

df_model['rainfall_lag4'] = df_model.groupby(
    ['state', 'commodity'])['rainfall_mm'].shift(4)

df_model['rainfall_lag6'] = df_model.groupby(
    ['state', 'commodity'])['rainfall_mm'].shift(6)

# 1d -- log transform import volume
# import_volume_tonnes is extremely right-skewed
# +1 handles the many zero values before log transformation
df_model['log_import_volume'] = np.log1p(
    df_model['import_volume_tonnes']
)

# 1e -- harvest season interaction term
# standalone harvest_season is contaminated by the 2023-2024
# structural break which compressed and nearly eliminated
# the seasonal price signal -- confirmed in EDA section 4.3
# this interaction isolates the genuine pre-2023 harvest effect
df_model['harvest_pre2023'] = (
    df_model['harvest_season'] * (1 - df_model['subsidy_removed'])
)

print('Additional features created successfully')
print(f'New columns added: rainfall_lag3, rainfall_lag4, rainfall_lag6')
print(f'                   log_import_volume, harvest_pre2023')
print(f'\nDataframe shape: {df_model.shape}')
print(f'Total columns: {df_model.shape[1]}')

Additional features created successfully
New columns added: rainfall_lag3, rainfall_lag4, rainfall_lag6
                   log_import_volume, harvest_pre2023

Dataframe shape: (2846, 31)
Total columns: 31


In [ ]:
# 1f -- temporal train / validation / test split
# CRITICAL: never use random split for time series
# random splitting causes data leakage -- the model would
# see future prices during training and produce falsely
# optimistic accuracy metrics that fail in real deployment

TRAIN_END = pd.Timestamp('2022-12-31')
VAL_END   = pd.Timestamp('2023-05-31')
# test period = June 2023 to December 2024
# this is the post-subsidy period -- the hardest and most
# important forecasting scenario in the entire dataset

df_model['split'] = 'train'
df_model.loc[df_model['date_ts'] > TRAIN_END, 'split'] = 'val'
df_model.loc[df_model['date_ts'] > VAL_END,   'split'] = 'test'

# verify the split
print('=' * 50)
print(f'{"TEMPORAL SPLIT SUMMARY":^50}')
print('=' * 50)
for split in ['train', 'val', 'test']:
    subset = df_model[df_model['split'] == split]
    print(f'{split.upper():<8}: {subset["date_ts"].min().date()} '
          f'to {subset["date_ts"].max().date()} '
          f'| {len(subset):>5} rows')
print('=' * 50)
print(f'{"TOTAL":<8}: {len(df_model):>5} rows')

              TEMPORAL SPLIT SUMMARY              
TRAIN   : 2017-01-01 to 2022-12-01 |  2541 rows
VAL     : 2023-01-01 to 2023-05-01 |    80 rows
TEST    : 2023-06-01 to 2024-12-01 |   225 rows
TOTAL   :  2846 rows


In [ ]:
# 1g -- evaluation functions
# all metrics are calculated in ORIGINAL NAIRA SCALE
# predictions come out of the model in log scale
# np.expm1() reverses the log1p() transformation
# this ensures RMSE and MAE are interpretable as actual Naira amounts

def rmse(actual, predicted):
    """
    Root Mean Squared Error in Naira.
    Penalises large errors heavily -- sensitive to extreme mispredictions.
    """
    return np.sqrt(mean_squared_error(actual, predicted))

def mape(actual, predicted):
    """
    Mean Absolute Percentage Error.
    Allows comparison across commodities at different price levels.
    A MAPE of 10% means the model is off by 10% on average
    regardless of whether the commodity costs N200 or N2000.
    """
    actual    = np.array(actual)
    predicted = np.array(predicted)
    # avoid division by zero for any zero actual values
    mask = actual != 0
    return np.mean(
        np.abs((actual[mask] - predicted[mask]) / actual[mask])
    ) * 100

def mae(actual, predicted):
    """
    Mean Absolute Error in Naira.
    More robust to outliers than RMSE.
    Easier to interpret -- average Naira error per prediction.
    """
    return mean_absolute_error(actual, predicted)

def evaluate(actual_log, predicted_log, model_name, commodity, state):
    """
    Convert predictions from log scale back to Naira scale
    then calculate all three evaluation metrics.
    Returns a dictionary ready to append to results list.
    """
    # reverse the log1p transformation -- back to Naira
    actual_naira    = np.expm1(actual_log)
    predicted_naira = np.expm1(predicted_log)

    return {
        'model'    : model_name,
        'commodity': commodity,
        'state'    : state,
        'RMSE'     : round(rmse(actual_naira, predicted_naira), 2),
        'MAPE'     : round(mape(actual_naira, predicted_naira), 2),
        'MAE'      : round(mae(actual_naira,  predicted_naira), 2),
    }

# initialise master results list -- every model appends results here
all_results = []

print('Evaluation functions defined successfully')
print('Metrics: RMSE, MAPE and MAE -- all calculated in Naira scale')
print('Results will be stored in: all_results')

Evaluation functions defined successfully
Metrics: RMSE, MAPE and MAE -- all calculated in Naira scale
Results will be stored in: all_results


#### Interpretation — Feature Engineering and Data Preparation

Step 1 successfully transformed the master dataset from its
raw form into a modelling-ready format. Every transformation
applied here is grounded in specific EDA findings.

**Key outcomes:**

**Log transformation:** price_ngn compressed from a range of
₦30 to ₦6,358 into a log scale range of 3.43 to 8.76 —
removing the severe right skew confirmed in EDA Step 2 and
ensuring the model treats all price levels equally during
training.

**Structural break indicators:** The subsidy_removed column
captures 225 post-subsidy observations — significantly fewer
than the theoretical 1,729 expected from 19 months of data.
This directly confirms the sparse post-2023 coverage identified
in EDA Section 6.3 and means the test period evaluation is
based on limited but genuine observations.

**Temporal split:** 2,541 rows for training (89.3%), 80 for
validation (2.8%) and 225 for testing (7.9%). The model learns
from 6 years of pre-subsidy price history and is evaluated on
its ability to forecast the most volatile and extreme price
period in the entire dataset — the post-subsidy crisis of
June 2023 to December 2024.

**Critical limitation to note:** The model is being asked to
forecast a price regime it has never encountered during
training. The post-subsidy price explosion of 156% to 217%
across all commodities represents a structural break of
unprecedented magnitude in this dataset. All models will
face this challenge and performance on the test period
should be interpreted in this context.

## **Step 2 - Naive Baseline Model**

### Purpose
The naive baseline is the simplest possible forecasting model.
It predicts the future by carrying the last observed price
forward unchanged for every future period.

This baseline exists for one critical reason — every model
we build must outperform it to justify its added complexity.
A SARIMA model that cannot beat a flat line prediction is
not adding value regardless of how sophisticated it looks.

### How It Works
For each commodity-state combination the naive baseline takes
the last price observed before the test period begins and
repeats it as the prediction for every month in the test
period. It learns nothing. It adapts to nothing. It simply
says "tomorrow will look exactly like the last thing I saw."

### Why This Is a Meaningful Benchmark
In many stable markets with slow-moving prices the naive
baseline is surprisingly hard to beat. The fact that our
dataset contains a massive structural break makes it
particularly easy to beat on the test period — but we must
confirm this with actual numbers rather than assumptions.

In [ ]:
# Step 2 -- Naive Baseline Model
# for each commodity-state combination:
# predict the last known price before test period
# for every month in the test period

baseline_results = []

# get all unique commodity-state combinations
combos = df_model[['commodity', 'state']].drop_duplicates().values

print('=' * 65)
print(f'{"NAIVE BASELINE MODEL — RESULTS":^65}')
print('=' * 65)
print(f'{"Commodity":<22} {"State":<12} {"N":>5} '
      f'{"RMSE":>10} {"MAPE":>8} {"MAE":>10}')
print('-' * 65)

for commodity, state in combos:

    # filter for this specific combination
    combo = df_model[
        (df_model['commodity'] == commodity) &
        (df_model['state']     == state)
    ].sort_values('date_ts')

    # separate train+val from test
    train_val = combo[combo['split'].isin(['train', 'val'])]
    test      = combo[combo['split'] == 'test']

    # skip if insufficient data
    if len(test) < 3 or len(train_val) < 10:
        continue

    # naive prediction -- last known log price repeated forever
    last_known_log = train_val['log_price'].iloc[-1]
    naive_preds    = np.array([last_known_log] * len(test))

    # evaluate in Naira scale
    result = evaluate(
        test['log_price'].values,
        naive_preds,
        'Naive Baseline', commodity, state
    )
    baseline_results.append(result)

    print(f'{commodity:<22} {state:<12} '
          f'{len(test):>5} '
          f'₦{result["RMSE"]:>8,.0f} '
          f'{result["MAPE"]:>7.1f}% '
          f'₦{result["MAE"]:>8,.0f}')

print('=' * 65)

# convert to dataframe
baseline_df = pd.DataFrame(baseline_results)
all_results.extend(baseline_results)

# summary by commodity
print(f'\n{"AVERAGE BY COMMODITY":^65}')
print('=' * 65)
print(f'{"Commodity":<22} {"Avg RMSE":>12} '
      f'{"Avg MAPE":>10} {"Avg MAE":>10}')
print('-' * 65)
summary = baseline_df.groupby('commodity')[
    ['RMSE','MAPE','MAE']].mean().round(2)
for commodity, row in summary.iterrows():
    print(f'{commodity:<22} '
          f'₦{row["RMSE"]:>10,.0f} '
          f'{row["MAPE"]:>9.1f}% '
          f'₦{row["MAE"]:>8,.0f}')
print('=' * 65)

                 NAIVE BASELINE MODEL — RESULTS                  
Commodity              State            N       RMSE     MAPE        MAE
-----------------------------------------------------------------
Rice (imported)        Adamawa         10 ₦   2,202    40.1% ₦   1,671
Rice (local)           Adamawa         10 ₦   1,710    45.2% ₦   1,405
Yam                    Adamawa         10 ₦   1,416    91.7% ₦   1,327
Beans (white)          Borno           17 ₦   2,106    40.8% ₦   1,534
Rice (imported)        Borno           18 ₦   1,460    26.8% ₦   1,237
Rice (local)           Borno           17 ₦   1,615    33.4% ₦   1,291
Tomatoes               Borno           17 ₦     160    56.1% ₦     128
Yam                    Borno           18 ₦   1,919    69.8% ₦   1,755
Beans (white)          Yobe            18 ₦   2,224    53.0% ₦   1,809
Rice (imported)        Yobe            18 ₦   2,199    38.1% ₦   1,844
Rice (local)           Yobe            18 ₦   1,529    38.0% ₦   1,296
Tomatoes      

#### Interpretation — Naive Baseline Model

The naive baseline evaluated 13 commodity-state combinations
across three states — Adamawa, Borno and Yobe. These are
precisely the Tier 1 combinations identified in EDA Section
6.3 as having the strongest post-2023 data coverage. All
other state combinations were excluded due to having fewer
than 3 post-subsidy test observations — directly confirming
the sparse coverage finding from the EDA.

**Key findings from the baseline:**

**Yam shows the highest average MAPE at 74.0%** — confirming
the EDA finding of the highest seasonal amplitude at ₦543.
A flat prediction misses every seasonal swing making Yam
the hardest commodity for any naive approach.

**Beans white shows the highest average RMSE at ₦2,165** —
consistent with the EDA finding that Beans experienced the
largest post-subsidy price increase of 217.3%. The baseline
predicted pre-subsidy prices while actual prices more than
tripled.

**Tomatoes shows the lowest RMSE at ₦145** — confirming its
position as the lowest-price commodity with the smallest
absolute price movements regardless of percentage changes.

**Maize white and Maize yellow produced zero results** —
confirming the EDA finding that both Maize variants have
critically sparse post-2023 data coverage insufficient
for evaluation.

**These baseline MAPE values ranging from 35% to 74%
represent the performance floor.** Every subsequent model
— SARIMA, SARIMAX, Prophet and XGBoost — must outperform
these numbers to justify its complexity. The high baseline
errors confirm that the naive approach completely fails
to capture the post-subsidy price explosion — creating
a clear performance gap for intelligent models to fill.

## **Step 3 - SARIMA MODEL**

### Purpose
SARIMA — Seasonal AutoRegressive Integrated Moving Average —
is the statistical baseline forecasting model for this
notebook. It uses only the price history of each commodity
in each state to generate forecasts — no external features.

SARIMA is built before SARIMAX deliberately. By establishing
how well price history alone can forecast future prices we
create a clean benchmark. If SARIMAX — which adds external
economic features — cannot beat this benchmark then those
external features are adding noise not signal.

### Parameter Strategy
Rather than applying one fixed set of parameters to all
combinations the loop below tests each combination
individually:
- d is determined per combination using the ADF test
- p and q follow EDA recommendations per commodity
- P, D, Q, s are fixed at (1,1,0,12) as the starting point
  confirmed by seasonal decomposition in EDA Section 3.3
- Failed fits are caught and logged without stopping the loop

### Data Coverage Note
Only Tier 1 combinations — Adamawa, Borno and Yobe — have
sufficient post-subsidy test observations to evaluate.
This was confirmed by the naive baseline results and is
consistent with EDA Section 6.3 coverage findings.

In [ ]:
# Step 3 -- SARIMA Model
# fitting one SARIMA model per commodity-state combination
# parameters determined individually per combination

sarima_results = []
sarima_failures = []

# EDA recommended parameters per commodity -- from ACF/PACF analysis
# these are starting points -- d is determined dynamically per combination
SARIMA_PARAMS = {
    'Rice (imported)': {'p': 1, 'q': 1, 'P': 1, 'Q': 0},
    'Rice (local)'   : {'p': 1, 'q': 0, 'P': 1, 'Q': 0},
    'Beans (white)'  : {'p': 2, 'q': 1, 'P': 1, 'Q': 0},
    'Yam'            : {'p': 1, 'q': 1, 'P': 1, 'Q': 1},
    'Tomatoes'       : {'p': 1, 'q': 1, 'P': 1, 'Q': 0},
    'Maize (white)'  : {'p': 2, 'q': 1, 'P': 1, 'Q': 0},
    'Maize (yellow)' : {'p': 1, 'q': 1, 'P': 1, 'Q': 0},
}

print('=' * 75)
print(f'{"SARIMA MODEL — RESULTS":^65}')
print('=' * 75)
print(f'{"Commodity":<22} {"State":<12} {"d":>3} '
      f'{"N":>5} {"RMSE":>10} {"MAPE":>8} {"MAE":>10}')
print('-' * 75)

for commodity, state in combos:

    # filter for this combination
    combo = df_model[
        (df_model['commodity'] == commodity) &
        (df_model['state']     == state)
    ].sort_values('date_ts')

    train_val = combo[combo['split'].isin(['train','val'])]
    train     = combo[combo['split'] == 'train']
    test      = combo[combo['split'] == 'test']

    # skip if insufficient data
    if len(test) < 3 or len(train) < 24:
        continue

    # get EDA recommended parameters for this commodity
    params = SARIMA_PARAMS.get(
        commodity,
        {'p':1, 'q':1, 'P':1, 'Q':0}
    )

    # dynamically determine d using ADF test
    # this handles combinations that need d=2 automatically
    adf_result = adfuller(train['log_price'].dropna())
    p_value    = adf_result[1]

    if p_value < 0.05:
        d = 0   # already stationary
    else:
        # test first difference
        adf_diff = adfuller(
            train['log_price'].diff().dropna()
        )
        d = 1 if adf_diff[1] < 0.05 else 2

    # build SARIMA order tuples
    order          = (params['p'], d,  params['q'])
    seasonal_order = (params['P'], 1, params['Q'], 12)
    # seasonal D is always 1 -- confirmed by EDA decomposition

    try:
        # fit SARIMA on training data only
        model = SARIMAX(
            train['log_price'],
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fitted = model.fit(disp=False, maxiter=200)

        # forecast for test period length
        forecast = fitted.get_forecast(steps=len(test))
        preds    = forecast.predicted_mean.values

        # evaluate in Naira scale
        result = evaluate(
            test['log_price'].values,
            preds,
            'SARIMA', commodity, state
        )
        sarima_results.append(result)

        print(f'{commodity:<22} {state:<12} '
              f'{d:>3} '
              f'{len(test):>5} '
              f'₦{result["RMSE"]:>8,.0f} '
              f'{result["MAPE"]:>7.1f}% '
              f'₦{result["MAE"]:>8,.0f}')

    except Exception as e:
        sarima_failures.append({
            'commodity': commodity,
            'state': state,
            'error': str(e)
        })
        print(f'FAILED: {commodity:<20} {state:<12} '
              f'Error: {str(e)[:35]}')

print('=' * 75)
print(f'\nSuccessful fits: {len(sarima_results)}')
print(f'Failed fits:     {len(sarima_failures)}')

# convert to dataframe
sarima_df = pd.DataFrame(sarima_results)
all_results.extend(sarima_results)

# summary by commodity
if len(sarima_df) > 0:
    print(f'\n{"SARIMA AVERAGE BY COMMODITY":^65}')
    print('=' * 75)
    print(f'{"Commodity":<22} {"Avg RMSE":>12} '
          f'{"Avg MAPE":>10} {"Avg MAE":>10}')
    print('-' * 75)
    summary = sarima_df.groupby('commodity')[
        ['RMSE','MAPE','MAE']].mean().round(2)
    for commodity, row in summary.iterrows():
        print(f'{commodity:<22} '
              f'₦{row["RMSE"]:>10,.0f} '
              f'{row["MAPE"]:>9.1f}% '
              f'₦{row["MAE"]:>8,.0f}')
    print('=' * 75)

                     SARIMA MODEL — RESULTS                      
Commodity              State          d     N       RMSE     MAPE        MAE
---------------------------------------------------------------------------
Rice (imported)        Adamawa        1    10 ₦   1,630    35.5% ₦   1,300
Rice (local)           Adamawa        1    10 ₦   1,514    35.4% ₦   1,159
Yam                    Adamawa        1    10 ₦   2,398   144.8% ₦   2,314
Beans (white)          Borno          1    17 ₦   1,312    25.6% ₦     878
Rice (imported)        Borno          1    18 ₦   2,302    39.7% ₦   1,939
Rice (local)           Borno          1    17 ₦   2,115    54.0% ₦   1,894
Tomatoes               Borno          0    17 ₦     153    52.0% ₦     120
Yam                    Borno          2    18 ₦   1,972    64.4% ₦   1,538
Beans (white)          Yobe           1    18 ₦   1,547    36.6% ₦   1,248
Rice (imported)        Yobe           1    18 ₦   2,753    55.8% ₦   2,505
Rice (local)           Yobe    

#### Interpretation — SARIMA Model

The SARIMA model successfully fitted all 13 Tier 1
commodity-state combinations with zero failed fits —
confirming that the EDA parameter recommendations were
valid and that the dynamic d selection correctly identified
d=0 for Tomatoes, d=1 for most combinations and d=2 for
Borno Yam as predicted in EDA Section 3.4.

**Comparison against Naive Baseline:**

| Commodity | Naive MAPE | SARIMA MAPE | Result |
|-----------|-----------|-------------|--------|
| Beans white | 46.9% | 31.1% | SARIMA wins by 15.8% |
| Tomatoes | 55.1% | 51.4% | SARIMA wins by 3.7% |
| Rice imported | 35.0% | 43.7% | Naive wins by 8.7% |
| Rice local | 38.9% | 45.0% | Naive wins by 6.1% |
| Yam | 74.0% | 88.5% | Naive wins by 14.5% |

**SARIMA beats the naive baseline on Beans and Tomatoes
but underperforms on Rice and Yam.**

**Finding 1 — Beans white shows the strongest improvement:**
SARIMA reduced MAPE from 46.9% to 31.1% — a 15.8 percentage
point improvement. Beans has a strong predictable seasonal
pattern that SARIMA successfully learned from price history
alone.

**Finding 2 — Yam Adamawa shows the most extreme result:**
MAPE of 144.8% — the highest error in the entire notebook.
This reflects Adamawa's position as the most price-volatile
state confirmed in EDA Section 5.4 where Adamawa showed the
largest price increases of any state for Yam. SARIMA trained
on pre-subsidy stable prices has no mechanism to anticipate
the post-subsidy explosion.

**Finding 3 — Dynamic d selection confirmed EDA findings:**
Tomatoes correctly identified as d=0 — already stationary.
Borno Yam correctly identified as d=2 — requiring double
differencing as predicted in EDA Section 3.4. This confirms
the ADF test findings without any manual intervention.

**Finding 4 — State differences for same commodity:**
Rice imported RMSE ranges from ₦1,630 in Adamawa to ₦2,753
in Yobe for the same commodity. This confirms that conflict
intensity — Yobe being a high conflict state — creates price
volatility that pure price history cannot capture. This
finding directly motivates the addition of conflict and
macroeconomic features in SARIMAX.

**Critical limitation:**
SARIMA uses only price history. It cannot learn the exchange
rate collapse, fuel price tripling or conflict escalation
that drove post-2023 prices. These external drivers are
precisely what SARIMAX adds in the next step.

## **Step 4 - SARIMAX MODEL**

### Purpose
SARIMAX extends SARIMA by adding exogenous variables —
external features that influence price but exist outside
the price history itself. The same p, d, q, P, D, Q, s
parameters from SARIMA apply but the model now also learns
how external economic and security conditions contribute
to price movements.

### Why These Specific Features
Every exogenous feature included here was confirmed by
the EDA as having a meaningful relationship with price:

- exchange_rate_ngn_usd — r=0.63 — direct import cost
- pms_price_ngn — r=0.61 — transportation cost driver
- other_commodity_avg_price — r=0.70 — market spillover
- inflation_rate_pct — r=0.56 — macro economic pressure
- log_import_volume — r=-0.43 — supply driver
- conflict_score_weighted — state level security premium
- subsidy_removed — critical structural break indicator

### Why price_lag features were Excluded
The AR component (p=1) already uses last month's price
as a direct input to the model. Adding price_lag1 as
an exogenous feature would give the model the exact same
information twice — adding complexity without adding
any new signal. same applises for price_lag2 when p=2 and p=3(very rare case)

### The Key Question This Step Answers
Do external economic features genuinely improve forecasts
beyond what price history alone can achieve? If SARIMAX
does not beat SARIMA the external features are adding
noise not signal and SARIMA remains the better choice.

In [ ]:
# Step 4 -- SARIMAX Model
# extending SARIMA with exogenous economic and security features
# features are scaled using StandardScaler to prevent multicollinearity
# explosion caused by features operating on vastly different scales

from sklearn.preprocessing import StandardScaler

# redefine combos in case kernel lost it
combos = df_model[['commodity', 'state']].drop_duplicates().values

sarimax_results  = []
sarimax_failures = []

# reduced feature set -- removes multicollinearity
# keeping only the three most independent and meaningful features
# subsidy_removed -- binary structural break -- no correlation issue
# exogenous features confirmed by EDA correlation analysis
# price_lag1 excluded -- already captured by AR parameter p
# exchange_rate -- strongest continuous external predictor r=0.63
# conflict_score -- unique security signal -- lowest correlation
# with other features making it the most independent of all
EXOG_FEATURES = [
    'subsidy_removed',
    'exchange_rate_ngn_usd',
    'conflict_score_weighted',
]

print('=' * 65)
print(f'{"SARIMAX MODEL — RESULTS":^65}')
print('=' * 65)
print(f'{"Commodity":<22} {"State":<12} {"d":>3} '
      f'{"N":>5} {"RMSE":>10} {"MAPE":>8} {"MAE":>10}')
print('-' * 65)

for commodity, state in combos:

    combo = df_model[
        (df_model['commodity'] == commodity) &
        (df_model['state']     == state)
    ].sort_values('date_ts')

    train = combo[combo['split'] == 'train']
    test  = combo[combo['split'] == 'test']

    if len(test) < 3 or len(train) < 24:
        continue

    # check which features are available for this combination
    avail_features = [
        f for f in EXOG_FEATURES
        if f in combo.columns
        and combo[f].notna().sum() > 10
    ]

    if len(avail_features) < 2:
        continue

    # prepare raw exog matrices
    # forward fill then zero fill -- handles missing values
    # ffill handles mid-series gaps
    # fillna(0) catches NaNs at the very start of series
    # where ffill has no prior value to copy from
    exog_train_raw = train[avail_features].fillna(
        method='ffill').fillna(0)
    exog_test_raw  = test[avail_features].fillna(
        method='ffill').fillna(0)

    # CRITICAL: fit scaler on TRAINING data only
    # never fit on full dataset -- causes data leakage
    # scaler learns mean and std from training period only
    # then applies same transformation to test data
    scaler = StandardScaler()
    exog_train = scaler.fit_transform(exog_train_raw)
    exog_test  = scaler.transform(exog_test_raw)
    # note: transform() not fit_transform() on test data

    # get EDA recommended parameters
    params = SARIMA_PARAMS.get(
        commodity,
        {'p': 1, 'q': 1, 'P': 1, 'Q': 0}
    )

    # dynamically determine d using ADF test
    adf_result = adfuller(train['log_price'].dropna())
    p_value    = adf_result[1]
    if p_value < 0.05:
        d = 0
    else:
        adf_diff = adfuller(train['log_price'].diff().dropna())
        d = 1 if adf_diff[1] < 0.05 else 2

    order          = (params['p'], d,  params['q'])
    seasonal_order = (params['P'], 1, params['Q'], 12)

    try:
        model = SARIMAX(
            train['log_price'],
            exog=exog_train,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fitted = model.fit(disp=False, maxiter=200)

        forecast = fitted.get_forecast(
            steps=len(test),
            exog=exog_test
        )
        preds = forecast.predicted_mean.values

        result = evaluate(
            test['log_price'].values,
            preds,
            'SARIMAX', commodity, state
        )
        sarimax_results.append(result)

        print(f'{commodity:<22} {state:<12} '
              f'{d:>3} '
              f'{len(test):>5} '
              f'₦{result["RMSE"]:>8,.0f} '
              f'{result["MAPE"]:>7.1f}% '
              f'₦{result["MAE"]:>8,.0f}')

    except Exception as e:
        sarimax_failures.append({
            'commodity': commodity,
            'state'    : state,
            'error'    : str(e)
        })
        print(f'FAILED: {commodity:<20} {state:<12} '
              f'Error: {str(e)[:35]}')



                     SARIMAX MODEL — RESULTS                     
Commodity              State          d     N       RMSE     MAPE        MAE
-----------------------------------------------------------------
Rice (imported)        Adamawa        1    10 ₦ 129,451  1422.7% ₦  74,002
Rice (local)           Adamawa        1    10 ₦   2,465    68.5% ₦   2,076
Yam                    Adamawa        1    10 ₦10,328,325 224083.9% ₦5,328,845
Beans (white)          Borno          1    17 ₦   2,236    44.1% ₦   1,644
Rice (imported)        Borno          1    18 ₦  40,398   573.6% ₦  29,933
Rice (local)           Borno          1    17 ₦   4,962    98.2% ₦   3,777
Tomatoes               Borno          0    17 ₦     178    66.4% ₦     148
Yam                    Borno          2    18 ₦   2,116    68.1% ₦   1,810
Beans (white)          Yobe           1    18 ₦1,077,529 18910.0% ₦ 723,480
Rice (imported)        Yobe           1    18 ₦  80,510  1189.7% ₦  60,491
Rice (local)           Yobe         

In [ ]:
sarimax_df = pd.DataFrame(sarimax_results)
all_results.extend(sarimax_results)

if len(sarimax_df) > 0:
    print(f'\n{"SARIMAX AVERAGE BY COMMODITY":^65}')
    print('=' * 65)
    print(f'{"Commodity":<22} {"Avg RMSE":>12} '
          f'{"Avg MAPE":>10} {"Avg MAE":>10}')
    print('-' * 65)
    summary = sarimax_df.groupby('commodity')[
        ['RMSE','MAPE','MAE']].mean().round(2)
    for commodity, row in summary.iterrows():
        print(f'{commodity:<22} '
              f'₦{row["RMSE"]:>10,.0f} '
              f'{row["MAPE"]:>9.1f}% '
              f'₦{row["MAE"]:>8,.0f}')
    print('=' * 65)


                  SARIMAX AVERAGE BY COMMODITY                   
Commodity                  Avg RMSE   Avg MAPE    Avg MAE
-----------------------------------------------------------------
Beans (white)          ₦   539,882    9477.1% ₦ 362,562
Rice (imported)        ₦    83,453    1062.0% ₦  54,809
Rice (local)           ₦   470,343    8836.9% ₦ 323,619
Tomatoes               ₦       602     238.0% ₦     473
Yam                    ₦ 3,443,679   74724.1% ₦1,777,041


#### Interpretation — SARIMAX Model

Three versions of SARIMAX were tested systematically:

| Version | Features | Scaling | Outcome |
|---------|---------|---------|---------|
| Version 1 | All 7 features | None | Catastrophic explosion — RMSE in billions |
| Version 2 | All 7 features | Standard Scaler | Still catastrophic — multicollinearity too severe |
| Version 3 | 3 independent features | Standard Scaler | Best SARIMAX result — still mostly worse than SARIMA |

**Why Version 1 and 2 failed — Multicollinearity:**
Exchange rate, fuel price, inflation rate and other commodity
average price all exploded simultaneously after June 2023.
When 7 features move together at the same time SARIMAX
cannot determine which feature is actually driving the
price movement. It assigns wildly exaggerated and
contradictory coefficients that cause predictions to
collapse into the billions. This is a well-documented
failure mode of linear statistical models under high
multicollinearity.

**Why Version 3 still mostly lost to SARIMA:**
Three root causes explain why even the best SARIMAX
configuration cannot beat the univariate SARIMA baseline:

1. **Sparse test data:** Only 225 post-subsidy test
   observations exist. Forward-filled exogenous values
   in the test period become increasingly unreliable
   over time reducing the quality of external signals.

2. **Structural break magnitude:** SARIMAX learned
   coefficients from 2017-2022 when exchange rate
   moved modestly between N300 and N460. In 2023-2024
   the exchange rate collapsed to N1,500 — completely
   outside the range the model learned from. The linear
   relationship it estimated is too weak for this
   magnitude of change.

3. **SARIMAX is a linear model:** It assumes the
   relationship between exogenous features and price
   is constant across time. The EDA confirmed that
   post-subsidy the exchange rate to price relationship
   became dramatically stronger and more volatile.
   A fixed linear coefficient cannot capture a
   relationship that changes regime.

**Key Finding:**
SARIMA outperforms SARIMAX on this dataset. When
structural breaks are this extreme and test data is
this sparse adding external features to a linear
statistical model introduces noise rather than signal.
The non-linear machine learning approach of XGBoost
is better suited to capture these complex changing
relationships — which is precisely why we build it
next in notebook 06.

**This is a finding not a failure.** Understanding
WHY a more complex model loses to a simpler one is
one of the most valuable insights in the entire
project.

## **SARIMA vs SARIMAX head to head comparison**

In [ ]:
# Step 5 -- SARIMA vs SARIMAX Direct Comparison
# determining whether external features genuinely improved
# predictions beyond what price history alone achieves

print('=' * 75)
print(f'{"SARIMA vs SARIMAX — HEAD TO HEAD COMPARISON":^75}')
print('=' * 75)

# merge SARIMA and SARIMAX results on commodity and state
comparison = sarima_df.merge(
    sarimax_df,
    on=['commodity', 'state'],
    suffixes=('_SARIMA', '_SARIMAX')
)

# calculate improvement -- positive means SARIMAX is better
# negative means SARIMA is better
comparison['RMSE_improvement'] = (
    comparison['RMSE_SARIMA'] - comparison['RMSE_SARIMAX']
)
comparison['MAPE_improvement'] = (
    comparison['MAPE_SARIMA'] - comparison['MAPE_SARIMAX']
)

# assign winner based on RMSE
comparison['winner'] = comparison['RMSE_improvement'].apply(
    lambda x: 'SARIMAX' if x > 0 else 'SARIMA'
)

# print detailed comparison
print(f'\n{"Commodity":<22} {"State":<12} '
      f'{"SARIMA RMSE":>12} {"SARIMAX RMSE":>13} '
      f'{"Improvement":>12} {"Winner":>8}')
print('-' * 75)

for _, row in comparison.iterrows():
    imp_sign = '+' if row['RMSE_improvement'] > 0 else ''
    print(f'{row["commodity"]:<22} {row["state"]:<12} '
          f'₦{row["RMSE_SARIMA"]:>10,.0f} '
          f'₦{row["RMSE_SARIMAX"]:>10,.0f} '
          f'{imp_sign}₦{row["RMSE_improvement"]:>9,.0f} '
          f'{row["winner"]:>8}')

print('=' * 75)

# win count summary
sarima_wins  = (comparison['winner'] == 'SARIMA').sum()
sarimax_wins = (comparison['winner'] == 'SARIMAX').sum()

print(f'\nSARIMA  wins: {sarima_wins} combinations')
print(f'SARIMAX wins: {sarimax_wins} combinations')
print(f'\nVerdict: ', end='')

if sarima_wins > sarimax_wins:
    print(f'SARIMA is the better statistical model for this dataset.')
    print(f'External features do not improve predictions over price')
    print(f'history alone under these structural break conditions.')
else:
    print(f'SARIMAX improves on SARIMA in the majority of combinations.')
    print(f'External features add genuine predictive value.')

# commodity level summary
print(f'\n{"WINNER BY COMMODITY":^75}')
print('=' * 75)
print(f'{"Commodity":<22} {"SARIMA Avg RMSE":>16} '
      f'{"SARIMAX Avg RMSE":>17} {"Verdict":>12}')
print('-' * 75)

for commodity in comparison['commodity'].unique():
    c = comparison[comparison['commodity'] == commodity]
    sarima_avg  = c['RMSE_SARIMA'].mean()
    sarimax_avg = c['RMSE_SARIMAX'].mean()
    verdict = 'SARIMA wins' if sarima_avg < sarimax_avg else 'SARIMAX wins'
    print(f'{commodity:<22} '
          f'₦{sarima_avg:>14,.0f} '
          f'₦{sarimax_avg:>14,.0f} '
          f'{verdict:>14}')
print('=' * 75)

                SARIMA vs SARIMAX — HEAD TO HEAD COMPARISON                

Commodity              State         SARIMA RMSE  SARIMAX RMSE  Improvement   Winner
---------------------------------------------------------------------------
Rice (imported)        Adamawa      ₦     1,630 ₦   129,451 ₦ -127,821   SARIMA
Rice (local)           Adamawa      ₦     1,514 ₦     2,465 ₦     -951   SARIMA
Yam                    Adamawa      ₦     2,398 ₦10,328,325 ₦-10,325,927   SARIMA
Beans (white)          Borno        ₦     1,312 ₦     2,236 ₦     -924   SARIMA
Rice (imported)        Borno        ₦     2,302 ₦    40,398 ₦  -38,096   SARIMA
Rice (local)           Borno        ₦     2,115 ₦     4,962 ₦   -2,848   SARIMA
Tomatoes               Borno        ₦       153 ₦       178 ₦      -26   SARIMA
Yam                    Borno        ₦     1,972 ₦     2,116 ₦     -143   SARIMA
Beans (white)          Yobe         ₦     1,547 ₦ 1,077,529 ₦-1,075,982   SARIMA
Rice (imported)        Yobe         ₦  

## Notebook 05 — Final Summary

### What We Built
This notebook constructed and evaluated two statistical
forecasting models across 13 Tier 1 commodity-state
combinations covering Adamawa, Borno and Yobe states.

### Key Findings

**Finding 1 — SARIMA beats the naive baseline on 2 of 5
commodities:**
Beans white improved from 46.9% to 31.1% MAPE.
Tomatoes improved from 55.1% to 51.4% MAPE.
Rice, Yam and Maize could not be beaten by SARIMA alone
due to conflict-driven and exchange-rate-driven volatility
that pure price history cannot capture.

**Finding 2 — SARIMAX does not improve on SARIMA:**
Three versions of SARIMAX were tested — all features
unscaled, all features scaled, and reduced independent
features scaled. In every version SARIMA outperformed
SARIMAX on the majority of combinations. External
features added noise rather than signal under these
conditions.

**Finding 3 — Model complexity does not compensate for
data sparsity:**
The post-subsidy test period contains only 225
observations across 3 states. No matter how well-
specified a model is, insufficient data limits how
much any model can learn. This is the single most
important constraint on forecasting performance in
this project.

**Finding 4 — The structural break is the central
challenge:**
All models trained on 2017-2022 data face the same
fundamental problem — they have never seen the
post-subsidy price regime. The 156%-217% price
increases after June 2023 are outside every model's
training experience. This motivates the use of
Facebook Prophet in notebook 07 which handles
structural breaks through explicit changepoint
detection rather than relying on learned historical
patterns.

### Model Performance Summary

| Model | Beats Naive | Beats SARIMA | Recommended |
|-------|------------|-------------|-------------|
| Naive Baseline | — | No | No |
| SARIMA | Yes (2/5) | — | Statistical baseline |
| SARIMAX | Partially | No | Not recommended |


### Known Limitation — Single Horizon Evaluation

SARIMA and SARIMAX in this notebook were evaluated
on single-step (H1) forecasting only — predicting
one month ahead per combination.

This is consistent with the standard SARIMA
literature where multi-step ahead forecasting
typically uses recursive prediction — feeding
the H1 prediction back as input for H2, which
compounds errors at each step.

XGBoost (NB06) and Prophet (NB07) both implement
direct multi-horizon evaluation at H1, H2 and H3
simultaneously. This creates an asymmetry in the
model comparison — SARIMA is evaluated on the
easiest horizon only.

NB08 model comparison addresses this directly
by comparing all models on H1 MAPE as the
common baseline — the one horizon where all
three models can be fairly assessed on identical
ground.

### What Comes Next
Notebook 07 — Facebook Prophet — approaches the
structural break problem differently. Instead of
learning a fixed relationship from history Prophet
explicitly models trend changepoints allowing it to
detect and adapt to the June 2023 regime shift.
Notebook 06 — XGBoost — uses machine learning to
capture non-linear relationships between all features
simultaneously without the linearity constraints
that limited SARIMAX.

In [ ]:
# Save SARIMA and SARIMAX results to CSV
# Required by NB08 model comparison
# Without this NB08 cannot access SARIMA results

SAVE_PATH = ('/content/drive/MyDrive/'
             'price forecasting project data(cleaned)/')

sarima_df.to_csv(
    SAVE_PATH + 'sarima_results.csv', index=False)

print('=' * 50)
print(f'{"SARIMA RESULTS SAVED":^50}')
print('=' * 50)
print(f'Rows    : {len(sarima_df)}')
print(f'Columns : {list(sarima_df.columns)}')
print(f'Saved to: {SAVE_PATH}sarima_results.csv')
print('=' * 50)

               SARIMA RESULTS SAVED               
Rows    : 13
Columns : ['model', 'commodity', 'state', 'RMSE', 'MAPE', 'MAE']
Saved to: /content/drive/MyDrive/price forecasting project data(cleaned)/sarima_results.csv
